# Step 6 — Benchmarking: Accuracy, Runtime, Convergence

We compare all three methods side-by-side:
- **Explicit** — no linear solve, but needs many time steps to stay stable
- **Implicit** — one tridiagonal solve per step, unconditionally stable, first-order in time
- **Crank-Nicolson** — one tridiagonal solve per step, unconditionally stable, second-order in time

Explicit uses the minimum stable N for each M. Implicit and CN use N = M.

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import numpy as np
import matplotlib.pyplot as plt

from src.solvers.explicit import solve_explicit, stability_limit
from src.solvers.implicit import solve_implicit
from src.solvers.crank_nicolson import solve_crank_nicolson
from src.black_scholes import bs_call_price

## Parameters

In [ ]:
K     = 100
r     = 0.05
sigma = 0.20
T     = 1.0
S_max = 3 * K
S0    = 100.0

exact = bs_call_price(S0, K, r, sigma, T)
print(f'Black-Scholes call price at S={S0}: {exact:.4f}')

## 1. Collect Errors and Runtimes

Each solver is timed over 10 runs and averaged to reduce noise.

In [ ]:
# Multiples of 3 so S0=100 falls exactly on S[M//3] with S_max=300 — no interpolation error
grid_sizes = [30, 60, 90, 120, 150, 180, 210, 300]
REPS = 10

results = {label: {'M': [], 'N': [], 'err': [], 'ms': []}
           for label in ['Explicit', 'Implicit', 'Crank-Nicolson']}

for M in grid_sizes:
    N_exp = stability_limit(S_max, K, r, sigma, T, M)
    solvers = [
        ('Explicit',       solve_explicit,       N_exp),
        ('Implicit',       solve_implicit,       M),
        ('Crank-Nicolson', solve_crank_nicolson, M),
    ]
    for label, fn, N in solvers:
        t0 = time.perf_counter()
        for _ in range(REPS):
            S, V = fn(S_max, K, r, sigma, T, M, N, 'call')
        elapsed_ms = (time.perf_counter() - t0) / REPS * 1000

        err = abs(V[M // 3] - exact)   # S[M//3] == 100.0 exactly
        results[label]['M'].append(M)
        results[label]['N'].append(N)
        results[label]['err'].append(err)
        results[label]['ms'].append(elapsed_ms)

# Print summary table
print(f"{'Method':16s} {'M':>5} {'N':>6} {'Error':>12} {'Time (ms)':>10}")
print('-' * 55)
for label in ['Explicit', 'Implicit', 'Crank-Nicolson']:
    for M, N, err, ms in zip(*results[label].values()):
        print(f"{label:16s} {M:>5} {N:>6} {err:>12.6f} {ms:>10.2f}")
    print()

## 2. Convergence — Error vs Grid Size

In [ ]:
plt.figure(figsize=(7, 4))
for label, marker in [('Explicit', 'o'), ('Implicit', 's'), ('Crank-Nicolson', '^')]:
    plt.loglog(results[label]['M'], results[label]['err'], marker + '-', label=label)
plt.xlabel('Stock grid size M')
plt.ylabel('Absolute error at S=100')
plt.title('Convergence — Error vs Grid Size')
plt.legend()
plt.grid(True, which='both', ls='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Runtime vs Grid Size

In [ ]:
plt.figure(figsize=(7, 4))
for label, marker in [('Explicit', 'o'), ('Implicit', 's'), ('Crank-Nicolson', '^')]:
    plt.plot(results[label]['M'], results[label]['ms'], marker + '-', label=label)
plt.xlabel('Stock grid size M')
plt.ylabel('Runtime (ms)')
plt.title('Runtime vs Grid Size')
plt.legend()
plt.grid(True, ls='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. Accuracy vs Runtime (Efficiency Frontier)

The most useful comparison: for a given compute budget, which method gives the lowest error?

In [ ]:
plt.figure(figsize=(7, 4))
for label, marker in [('Explicit', 'o'), ('Implicit', 's'), ('Crank-Nicolson', '^')]:
    plt.loglog(results[label]['ms'], results[label]['err'], marker + '-', label=label)
plt.xlabel('Runtime (ms)')
plt.ylabel('Absolute error at S=100')
plt.title('Accuracy vs Runtime — Efficiency Frontier')
plt.legend()
plt.grid(True, which='both', ls='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 5. Summary

| Method | Stability | Time order | Space order | Notes |
|---|---|---|---|---|
| Explicit | Conditional | O(dt) | O(dS²) | Fast per step; needs many steps |
| Implicit | Unconditional | O(dt) | O(dS²) | One tridiagonal solve per step |
| Crank-Nicolson | Unconditional | O(dt²) | O(dS²) | Same cost as implicit; more accurate at fine grids |